# Pythia-410M: financial continued pretraining on a Colab NVIDIA GPU

This notebook adapts the **EleutherAI/pythia-410m base model** to financial prose using
Mixed-precision LoRA uses BF16 when natively supported and FP16 otherwise.
It does not use 4-bit quantization. This is text continuation, not
instruction tuning.

Select **Runtime > Change runtime type > GPU** (A100, L4, or T4), run installation, restart
the runtime if packages were already imported, then run the remaining cells in order.
Google Drive stores checkpoints, adapters, metrics, and a shared document sample.
The selected raw financial text is also cached in your own Drive for repeatable splits.

Defaults: 3,000 sampled documents, 1,024-token blocks, automatic microbatch sizing,
effective batch 8, and 100 optimizer steps. This is a pilot, not a full-corpus run.
For a smoke run use MAX_STEPS=2, EVAL_BLOCKS=4, EVAL_EVERY=2, SAVE_EVERY=2.

Both new model notebooks reuse the exact same saved document split. This split
does not reproduce the earlier scratch-model or Qwen notebook's split; those runs
must be aligned separately before a project-wide comparison. Compare each model
before/after adaptation on identical held-out text. Per-token perplexities from
different tokenizers are not directly comparable.

Local validation covers syntax and synthetic data/collator checks. A Colab GPU
training, save, and reload run remains necessary; no GPU results are prepopulated.


## 1. Install the notebook stack

Keep Colab's CUDA-enabled PyTorch. The notebook uses the pinned TRL/PEFT APIs below; restart after installation if these libraries were already loaded.

In [ ]:
import subprocess, sys
PACKAGES = [
    "transformers==4.51.3", "trl==0.17.0", "peft==0.15.2",
    "accelerate==1.6.0", "datasets==3.5.0",
    "huggingface_hub==0.30.2", "numpy<3",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])
print("Installation complete. If these libraries were already imported, restart the session now.")


## 2. Configuration and GPU checks

Choose a CUDA GPU runtime: A100, L4, T4, or another supported NVIDIA GPU.
Native BF16 support selects BF16; otherwise FP16 is used with Trainer loss scaling.
MICRO_BATCH=None chooses a conservative memory-based default. Override it with a
positive divisor of EFFECTIVE_BATCH (default 8); GRAD_ACCUM is calculated automatically.
For OOM, reduce MICRO_BATCH and rebuild the model/trainer.
For a smoke run set MAX_STEPS=2, EVAL_BLOCKS=4, EVAL_EVERY=2, SAVE_EVERY=2.


In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
import gc, math, json, random, hashlib, inspect, itertools, unicodedata, subprocess, sys
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import version
import torch
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi, login
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTConfig, SFTTrainer

MODEL_ID = "EleutherAI/pythia-410m"
EXPECTED_MODEL_TYPE = "gpt_neox"
LORA_TARGETS = ["query_key_value","dense","dense_h_to_4h","dense_4h_to_h"]
MICRO_BATCH = None
EFFECTIVE_BATCH = 8
DEFAULT_LARGE_GPU_BATCH = 4
DATASET_ID = "alwaysgood/financial-english-source-corpus"
MODEL_REVISION = "main"  # Resolved to an immutable SHA below; paste a SHA to reproduce.
DATASET_REVISION = "main"
DATASET_CONFIG = None
DATASET_SPLIT = "train"
TEXT_FIELD = "source"   # None allows unambiguous inference from source/text/content/document.
SEED = 42
SAMPLE_DOCS = 3000
MAX_SCANNED_ROWS = 30000
SHUFFLE_BUFFER = 1000
MIN_CHARS = 80
MAX_DOC_CHARS = 32768  # Explicit memory bound; long documents are truncated and counted.
BLOCK_SIZE = 1024
EVAL_FRACTION = 0.05
EVAL_BLOCKS = 64
MAX_STEPS = 100
LEARNING_RATE = 1e-4
EVAL_EVERY = 50
SAVE_EVERY = 50
LORA_R = 16
LORA_ALPHA = 32
DRIVE_ROOT = "/content/drive/MyDrive/pythia_410m_financial_lora"
RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S_%f")

if not torch.cuda.is_available():
    raise RuntimeError("Select an NVIDIA GPU in Runtime → Change runtime type, then rerun.")
assert 128 <= BLOCK_SIZE <= 4096
assert SAMPLE_DOCS >= 100 and MAX_SCANNED_ROWS >= SAMPLE_DOCS
assert 0 < EVAL_FRACTION < 0.5 and EVAL_BLOCKS >= 1
assert MAX_STEPS > 0 and EVAL_EVERY > 0 and SAVE_EVERY > 0
assert SHUFFLE_BUFFER > 0 and MAX_DOC_CHARS >= MIN_CHARS > 0
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float16
if MICRO_BATCH is None:
    MICRO_BATCH = DEFAULT_LARGE_GPU_BATCH if BF16 and GPU_MEMORY_GIB >= 35 else 1
if not isinstance(MICRO_BATCH, int) or MICRO_BATCH <= 0:
    raise ValueError("MICRO_BATCH must be a positive integer or None.")
if EFFECTIVE_BATCH <= 0 or EFFECTIVE_BATCH % MICRO_BATCH:
    raise ValueError("EFFECTIVE_BATCH must be positive and divisible by MICRO_BATCH.")
GRAD_ACCUM = EFFECTIVE_BATCH // MICRO_BATCH
set_seed(SEED)
versions = {p: version(p) for p in ["torch", "transformers", "trl", "peft", "datasets", "accelerate", "huggingface_hub"]}
print(versions)
print("Microbatch:", MICRO_BATCH, "| Gradient accumulation:", GRAD_ACCUM)
print("GPU:", torch.cuda.get_device_name(0), "| compute:", COMPUTE_DTYPE)
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Effective sequences/step:", MICRO_BATCH * GRAD_ACCUM, "| tokens/step:", MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE)
if version("transformers") != "4.51.3" or version("trl") != "0.17.0":
    raise RuntimeError("Pinned libraries are not active. Restart the session after installation.")


## 3. Mount Drive and resolve immutable Hub revisions

Each run uses a new directory. Checkpoints, the final LoRA adapter, tokenizer,
metrics, configuration, and document-hash split manifest are saved there.
The final adapter needs the original base model; it is not a standalone full model.


In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None  # A missing/denied optional secret is fine for these public repos.
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
del hf_token

api = HfApi()
MODEL_SHA = api.model_info(MODEL_ID, revision=MODEL_REVISION).sha
DATASET_SHA = api.dataset_info(DATASET_ID, revision=DATASET_REVISION).sha
RUN_DIR = Path(DRIVE_ROOT) / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
ADAPTER_DIR = RUN_DIR / "adapter"
def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
config = {k: v for k, v in list(globals().items()) if k.isupper() and isinstance(v, (str, int, float, bool, list, type(None)))}
config.update(model_sha=MODEL_SHA, dataset_sha=DATASET_SHA, versions=versions,
              run_dir=str(RUN_DIR), adapter_dir=str(ADAPTER_DIR))
save_json(RUN_DIR / "run_config.json", config)
print("Run folder:", RUN_DIR)
print("Model SHA:", MODEL_SHA, "| Dataset SHA:", DATASET_SHA)


## 4. Load or create the shared document sample

The first notebook creates a shared JSON cache in Drive. The other notebook
reuses its dataset revision, sampled texts, normalization, and train/eval membership.
A settings mismatch stops execution instead of silently using a different sample.
Do not run the cache-creation cells concurrently in two sessions.

Sampling uses a bounded shuffle buffer, not a globally uniform sample. Inspect
the source histogram: a narrow sample is only a pilot. Deduplication removes exact
normalized duplicates, not related excerpts. The evaluation split is validation,
not an independent final test. Do not publish cached corpus text without reviewing
the source terms.


In [ ]:
SHARED_PATH = Path("/content/drive/MyDrive/llm_flow_shared/financial_sample_v1.json")
sample_settings = dict(
    dataset_id=DATASET_ID, dataset_config=DATASET_CONFIG, dataset_split=DATASET_SPLIT,
    seed=SEED, sample_docs=SAMPLE_DOCS, max_scanned_rows=MAX_SCANNED_ROWS,
    shuffle_buffer=SHUFFLE_BUFFER, min_chars=MIN_CHARS,
    max_doc_chars=MAX_DOC_CHARS, text_field=TEXT_FIELD, eval_fraction=EVAL_FRACTION,
)
if SHARED_PATH.is_file():
    shared = json.loads(SHARED_PATH.read_text())
    if shared["settings"] != sample_settings:
        raise ValueError("Shared sample settings differ. Use the same settings or a new shared-cache filename.")
    DATASET_SHA = shared["dataset_sha"]
    docs = shared["documents"]
    sample_stats = Counter(shared["sampling"])
    selected_field = shared["text_field"]
    config["dataset_sha"] = DATASET_SHA
    save_json(RUN_DIR / "run_config.json", config)
else:
    stream = load_dataset(
        DATASET_ID, name=DATASET_CONFIG, split=DATASET_SPLIT,
        revision=DATASET_SHA, streaming=True,
    )
    print("Features:", stream.features)
    preview = list(itertools.islice(iter(stream), 5))
    if not preview:
        raise ValueError("The selected dataset split is empty.")
    print("Columns:", list(preview[0]))
    print("First-row types:", {k: type(v).__name__ for k, v in preview[0].items()})
    
    def select_text_field(rows, requested):
        columns = set(rows[0])
        if requested is not None:
            if requested not in columns:
                raise ValueError(f"TEXT_FIELD={requested!r} is absent. Columns: {sorted(columns)}")
            if not any(isinstance(r.get(requested), str) and r[requested].strip() for r in rows):
                raise TypeError(f"{requested!r} has no nonempty string in the preview. Inspect schema before training.")
            return requested
        choices = [k for k in ("source", "text", "content", "document")
                   if k in columns and any(isinstance(r.get(k), str) and r[k].strip() for r in rows)]
        if len(choices) != 1:
            raise ValueError(f"Ambiguous/missing text field: {choices}. Set TEXT_FIELD explicitly.")
        return choices[0]
    
    def clean_text(value):
        if not isinstance(value, str):
            return None, False
        value = unicodedata.normalize("NFKC", value).replace("\x00", "").strip()
        capped = len(value) > MAX_DOC_CHARS
        value = value[:MAX_DOC_CHARS].strip()
        if len(value) < MIN_CHARS:
            return None, capped
        return value, capped
    
    def text_hash(text):
        return hashlib.sha256(" ".join(text.split()).encode("utf-8")).hexdigest()
    
    selected_field = select_text_field(preview, TEXT_FIELD)
    print("Training text field:", selected_field)
    preview_text = next(r[selected_field] for r in preview if isinstance(r.get(selected_field), str) and r[selected_field].strip())
    print("Preview (first 300 characters):", repr(preview_text[:300]))
    docs, seen = [], set()
    sample_stats = Counter()
    shuffled = stream.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER)
    for row in itertools.islice(shuffled, MAX_SCANNED_ROWS):
        sample_stats["scanned"] += 1
        if selected_field not in row:
            raise ValueError("Inconsistent schema: selected text field missing in a streamed row.")
        text, capped = clean_text(row[selected_field])
        sample_stats["character_capped"] += int(capped)
        if text is None:
            sample_stats["invalid_or_short"] += 1
            continue
        digest = text_hash(text)
        if digest in seen:
            sample_stats["duplicates"] += 1
            continue
        seen.add(digest)
        docs.append({"text": text, "sha256": digest, "source_label": str(row.get("dataset", "unknown"))})
        if len(docs) == SAMPLE_DOCS:
            break
    if len(docs) < 100:
        raise ValueError(f"Only {len(docs)} usable documents. Inspect the schema or raise MAX_SCANNED_ROWS.")
    if len(docs) < SAMPLE_DOCS:
        print(f"Warning: scan ended with {len(docs)} of {SAMPLE_DOCS} requested documents.")
    print("Sampling:", dict(sample_stats), "| accepted:", len(docs))
    print("Source labels:", dict(Counter(d["source_label"] for d in docs)))
    
    if len(docs) < SAMPLE_DOCS:
        raise ValueError("Incomplete sample: increase MAX_SCANNED_ROWS before saving a shared cache.")
    membership = list(range(len(docs)))
    random.Random(SEED).shuffle(membership)
    n_eval = max(16, round(len(docs) * EVAL_FRACTION))
    eval_indices = set(membership[:n_eval])
    for i, doc in enumerate(docs):
        doc["split"] = "eval" if i in eval_indices else "train"
    shared = dict(settings=sample_settings, dataset_sha=DATASET_SHA,
                  documents=docs, sampling=dict(sample_stats), text_field=selected_field)
    SHARED_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary = SHARED_PATH.with_suffix(".tmp")
    save_json(temporary, shared)
    temporary.replace(SHARED_PATH)
if len(docs) != SAMPLE_DOCS or any(d.get("split") not in ("train", "eval") for d in docs):
    raise ValueError("Invalid shared document cache.")
for doc in docs:
    digest = hashlib.sha256(" ".join(doc["text"].split()).encode("utf-8")).hexdigest()
    if digest != doc["sha256"]:
        raise ValueError("Shared document hash mismatch.")
if len({d["sha256"] for d in docs}) != len(docs):
    raise ValueError("Duplicate hashes in shared document cache.")
config["shared_sample_sha256"] = hashlib.sha256(SHARED_PATH.read_bytes()).hexdigest()
config["shared_sample_path"] = str(SHARED_PATH)
save_json(RUN_DIR / "run_config.json", config)
print("Shared sample:", SHARED_PATH)
print("Source labels:", dict(Counter(d["source_label"] for d in docs)))
print("Cache SHA256:", config["shared_sample_sha256"])


## 5. Split documents, tokenize, and pack each split independently

The split happens **before** tokenization/packing, after exact deduplication.
Every document receives its tokenizer EOS token. Fixed-length blocks concatenate
documents within one split, so attention can cross EOS boundaries (standard
causal pretraining). Each split drops at most `BLOCK_SIZE - 1` trailing tokens.
Evaluation uses a fixed shuffled subset of at most `EVAL_BLOCKS` blocks.
No chat template or assistant-only loss mask is applied.


In [ ]:
train_docs = [d for d in docs if d["split"] == "train"]
eval_docs = [d for d in docs if d["split"] == "eval"]
assert set(d["sha256"] for d in train_docs).isdisjoint(d["sha256"] for d in eval_docs)
split_manifest = {"train": [{k: d[k] for k in ("sha256", "source_label")} for d in train_docs],
                  "eval": [{k: d[k] for k in ("sha256", "source_label")} for d in eval_docs]}
save_json(RUN_DIR / "split_manifest.json", split_manifest)
print("Documents:", len(train_docs), "train /", len(eval_docs), "eval")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_SHA, use_fast=True)
if tokenizer.eos_token_id is None:
    raise ValueError("A valid EOS token is required for document boundaries.")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def pack_documents(documents, tok, block_size):
    blocks, pending = [], []
    tokens = 0
    for doc in documents:
        ids = tok(doc["text"], add_special_tokens=False, truncation=False, verbose=False)["input_ids"]
        ids = ids + [tok.eos_token_id]
        tokens += len(ids)
        pending.extend(ids)
        offset = 0
        while len(pending) - offset >= block_size:
            blocks.append(pending[offset:offset + block_size])
            offset += block_size
        pending = pending[offset:]
    if not blocks:
        raise ValueError("Not enough tokens for one block. Increase the sample or lower BLOCK_SIZE.")
    return Dataset.from_dict({"input_ids": blocks}), {"tokens": tokens, "blocks": len(blocks), "tail_tokens_dropped": len(pending)}

train_data, train_token_stats = pack_documents(train_docs, tokenizer, BLOCK_SIZE)
eval_data, eval_token_stats = pack_documents(eval_docs, tokenizer, BLOCK_SIZE)
eval_data = eval_data.shuffle(seed=SEED).select(range(min(EVAL_BLOCKS, len(eval_data))))
print("Train:", train_token_stats, "| Eval:", eval_token_stats)
print("Evaluated blocks:", len(eval_data), "| Approximate training passes:", round(MAX_STEPS * MICRO_BATCH * GRAD_ACCUM / len(train_data), 2))
data_stats = {"sampling": dict(sample_stats), "train": train_token_stats, "eval": eval_token_stats,
              "eval_blocks_used": len(eval_data), "text_field": selected_field}
save_json(RUN_DIR / "data_stats.json", data_stats)
# Keep raw corpus text out of saved artifacts; hashes identify the split.
del docs, train_docs, eval_docs, shared
for variable in ("preview", "stream", "shuffled", "seen"):
    globals().pop(variable, None)
gc.collect()


## 6. Load the frozen mixed-precision base and attach LoRA adapters

This notebook uses ordinary LoRA with a BF16/FP16 base on one CUDA GPU. Only adapters
are trainable. Projection targets are selected for gpt_neox, checked at runtime,
and recorded in the run configuration. Gradient checkpointing reduces activations.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_SHA, torch_dtype=COMPUTE_DTYPE,
    device_map={"": 0}, attn_implementation="sdpa",
)
if model.config.model_type != EXPECTED_MODEL_TYPE:
    raise ValueError(f"Unexpected architecture: {model.config.model_type}")
if BLOCK_SIZE > model.config.max_position_embeddings:
    raise ValueError("BLOCK_SIZE exceeds the model context limit.")
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
missing = [name for name in LORA_TARGETS
           if not any(n.endswith("." + name) for n, _ in model.named_modules())]
if missing:
    raise ValueError(f"LoRA projection modules missing: {missing}")
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=LORA_TARGETS,
)
model = get_peft_model(model, lora_config)
for parameter in model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.to(torch.float32)
model.enable_input_require_grads()
model.print_trainable_parameters()
assert any(p.requires_grad for p in model.parameters())
assert all("lora_" in name for name, p in model.named_parameters() if p.requires_grad)


## 7. Build a version-aware TRL trainer and check one batch

The collator copies token IDs to labels; the model shifts them internally for
next-token prediction. Real EOS tokens remain supervised even when pad equals
EOS. These blocks have no padding. The general padding path masks positions by
attention mask, not token value. Evaluation returns loss only to avoid storing
vocabulary-sized logits. Training/evaluation logs stay local and in Drive.


In [ ]:
class CausalLMCollator:
    def __init__(self, tok):
        self.tok = tok
    def __call__(self, examples):
        batch = self.tok.pad(
            [{"input_ids": x["input_ids"], "attention_mask": [1] * len(x["input_ids"])} for x in examples],
            padding=True, return_tensors="pt",
        )
        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100
        batch["labels"] = labels
        return batch

collator = CausalLMCollator(tokenizer)
test_batch = collator([train_data[0]])
assert test_batch["input_ids"].shape == (1, BLOCK_SIZE)
assert torch.equal(test_batch["labels"], test_batch["input_ids"])
del test_batch

cfg_fields = inspect.signature(SFTConfig).parameters
trainer_fields = inspect.signature(SFTTrainer.__init__).parameters
cfg_kwargs = dict(
    output_dir=str(CHECKPOINT_DIR), max_steps=MAX_STEPS,
    per_device_train_batch_size=MICRO_BATCH, per_device_eval_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE, lr_scheduler_type="cosine", warmup_ratio=0.05,
    weight_decay=0.01, max_grad_norm=1.0,
    bf16=BF16, fp16=not BF16,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch", logging_steps=5, logging_first_step=True,
    eval_steps=EVAL_EVERY, save_strategy="steps", save_steps=SAVE_EVERY,
    save_total_limit=2, load_best_model_at_end=False,
    prediction_loss_only=True, report_to="none", push_to_hub=False,
    seed=SEED, data_seed=SEED, dataloader_num_workers=0,
    remove_unused_columns=False, label_names=["labels"], packing=False,
)
strategy_key = "eval_strategy" if "eval_strategy" in cfg_fields else "evaluation_strategy"
cfg_kwargs[strategy_key] = "steps"
length_key = "max_length" if "max_length" in cfg_fields else "max_seq_length"
if length_key not in cfg_fields or "dataset_kwargs" not in cfg_fields:
    raise RuntimeError("Unsupported TRL API. Reinstall the pinned stack and restart the session.")
cfg_kwargs[length_key] = BLOCK_SIZE
cfg_kwargs["dataset_kwargs"] = {"skip_prepare_dataset": True}
for name in ("completion_only_loss", "assistant_only_loss", "padding_free"):
    if name in cfg_fields:
        cfg_kwargs[name] = False
unsupported = set(cfg_kwargs) - set(cfg_fields)
if unsupported:
    raise RuntimeError(f"Unsupported training arguments {unsupported}; restore the pinned stack.")
training_args = SFTConfig(**cfg_kwargs)
trainer_kwargs = dict(model=model, args=training_args, train_dataset=train_data,
                      eval_dataset=eval_data, data_collator=collator)
if "processing_class" in trainer_fields:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_fields:
    trainer_kwargs["tokenizer"] = tokenizer
else:
    raise RuntimeError("Unsupported SFTTrainer signature. Restore the pinned stack.")
trainer = SFTTrainer(**trainer_kwargs)
# Drop this temporary dictionary so it cannot keep the GPU model alive at reload.
del trainer_kwargs
print("Trainer ready:", type(trainer).__name__, "|", length_key)

smoke_batch = {k: v.to("cuda:0") for k, v in collator([train_data[0]]).items()}
model.train()
with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
    smoke_loss = model(**smoke_batch).loss
if not torch.isfinite(smoke_loss):
    raise RuntimeError("Non-finite smoke loss.")
smoke_loss.backward()
adapter_grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
if not adapter_grads or not all(torch.isfinite(g).all().item() for g in adapter_grads):
    raise RuntimeError("Missing or non-finite adapter gradients.")
model.zero_grad(set_to_none=True)
del smoke_batch, smoke_loss, adapter_grads
print("GPU forward/backward smoke check passed.")


## 8. Evaluate the base, train, and evaluate the adapter

Baseline evaluation disables adapters on the same quantized base. Both scores
use the identical held-out blocks. Perplexity is `exp(loss)` on their supervised
next-token targets; lower is better. It measures prose prediction, not financial
reasoning or factual accuracy. This validation split is not a separate final test.


In [ ]:
def add_perplexity(metrics, prefix):
    result = dict(metrics)
    loss = float(result[f"{prefix}_loss"])
    if not math.isfinite(loss):
        raise RuntimeError(f"Non-finite {prefix} loss. Inspect data, GPU precision, and learning rate.")
    result[f"{prefix}_perplexity"] = math.exp(loss) if loss < 700 else None
    return result

with model.disable_adapter():
    baseline_metrics = add_perplexity(trainer.evaluate(metric_key_prefix="baseline"), "baseline")
save_json(RUN_DIR / "baseline_metrics.json", baseline_metrics)
print("Base:", baseline_metrics)

train_result = trainer.train()
final_metrics = add_perplexity(trainer.evaluate(metric_key_prefix="eval"), "eval")
comparison = {
    "baseline": baseline_metrics, "adapted": final_metrics,
    "loss_change": final_metrics["eval_loss"] - baseline_metrics["baseline_loss"],
    "training": train_result.metrics,
    "model_id": MODEL_ID, "model_sha": MODEL_SHA,
    "shared_sample_sha256": config["shared_sample_sha256"],
    "processed_token_budget": MAX_STEPS * MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE,
}
save_json(RUN_DIR / "metrics.json", comparison)
print(json.dumps(comparison, indent=2))


## 9. Save the adapter and tokenizer to Drive

The adapter needs the original base model and exact revision. Intermediate checkpoints retain optimizer state; the adapter directory is for inference. Save/load verification follows.

In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
trainer.save_state()
save_json(RUN_DIR / "trainer_log.json", trainer.state.log_history)
save_json(ADAPTER_DIR / "run_config.json", config)
installed = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_DIR / "requirements_freeze.txt").write_text(installed)
for filename in ("adapter_config.json", "adapter_model.safetensors", "tokenizer_config.json"):
    if not (ADAPTER_DIR / filename).is_file():
        raise RuntimeError(f"Expected saved file is missing: {filename}")
print("Saved final adapter:", ADAPTER_DIR)
print("Saved run artifacts:", sorted(p.name for p in RUN_DIR.iterdir()))


## 10. Reload the saved adapter and generate

Free the training model, load a fresh BF16/FP16 base and the saved adapter, then generate a plain-text continuation. In a new session, install dependencies, run imports/GPU checks, mount Drive, and replace ADAPTER_PATH with your saved directory.

In [ ]:
ADAPTER_PATH = str(ADAPTER_DIR)  # After a restart, replace with "/content/drive/MyDrive/.../adapter".
for variable in ("trainer", "model", "inference_model", "inference_base"):
    globals().pop(variable, None)
gc.collect()
torch.cuda.empty_cache()

saved_config = json.loads((Path(ADAPTER_PATH) / "run_config.json").read_text())
inference_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
if inference_tokenizer.pad_token_id is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token
inference_tokenizer.padding_side = "left"
inference_base = AutoModelForCausalLM.from_pretrained(
    saved_config["MODEL_ID"], revision=saved_config["model_sha"],
    torch_dtype=COMPUTE_DTYPE,
    device_map={"": 0}, attn_implementation="sdpa",
)
inference_model = PeftModel.from_pretrained(inference_base, ADAPTER_PATH, is_trainable=False)
inference_model.eval()
inference_model.config.use_cache = True

prompt = "The company's operating cash flow increased during the quarter, primarily because"
inputs = inference_tokenizer(prompt, return_tensors="pt").to("cuda:0")
with torch.inference_mode():
    generated = inference_model.generate(
        **inputs, max_new_tokens=150, do_sample=False, repetition_penalty=1.05,
        eos_token_id=inference_tokenizer.eos_token_id,
        pad_token_id=inference_tokenizer.pad_token_id, use_cache=True,
    )
print(inference_tokenizer.decode(generated[0], skip_special_tokens=True))


## Run notes and comparison protocol

- Start with MAX_STEPS=2, EVAL_BLOCKS=4, EVAL_EVERY=2, SAVE_EVERY=2.
  Verify finite loss, saved adapter files, and successful reload before a longer run.
- The same sample-cache hash and split manifest establish shared document membership.
  Keep the same sample settings across notebooks. To regenerate, choose a new shared
  cache filename in both notebooks. Original scratch/Qwen splits are not retroactively aligned.
- Both notebooks use the same nominal processed-token budget. Different tokenizers
  mean different quantities of source text per block. Report tokens, document passes,
  wall time, and adapter parameter counts; do not claim identical compute.
- Compare within-model baseline/adapted loss and identical external financial tasks.
  Do not rank different tokenizers by raw perplexity alone.
- Source coverage and near-duplicate leakage remain limitations of this pilot sample.
  A final study needs a separate, shared test split.
- Drive stores intermediate checkpoints every SAVE_EVERY steps. To resume, rebuild the
  exact saved model revision, shared sample, and configuration, then call
  trainer.train(resume_from_checkpoint=".../checkpoints/checkpoint-50").
  Do not use adapter/ as an optimizer checkpoint. Automatic resume is not implemented.
- Native BF16 support selects BF16; T4 uses FP16. Set MICRO_BATCH=1 if needed;
  GRAD_ACCUM is recalculated to preserve effective batch size. No bitsandbytes is required.
- No GPU training has been executed as part of local notebook creation.

References:
[model card](https://huggingface.co/EleutherAI/pythia-410m),
[TRL 0.17](https://huggingface.co/docs/trl/v0.17.0/sft_trainer),
[PEFT LoRA](https://huggingface.co/docs/peft/v0.15.0/developer_guides/lora),
[financial dataset](https://huggingface.co/datasets/alwaysgood/financial-english-source-corpus).
